# Creating LSTM Models

## Importing Libraries 

In [40]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

## Downloading Historical Stock Data


In [ ]:
# === Step 0: Define Top 10 Stocks ===
top_10_symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JPM']

In [ ]:
# === Step 1–9: Loop over all stocks ===
start_date = '2000-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')
time_steps = 60

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i - time_steps:i])
        y.append(data[i])
    return np.array(X), np.array(y)

def print_metrics_inline(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")

for directory in ["models", "datasets", "test_set_predictions", "training_loss", "metrics"]:
    os.makedirs(directory, exist_ok=True)

for symbol in top_10_symbols:
    print(f"\n=== Processing {symbol} ===")

    df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
    df = df[['Date', 'Close']].dropna()
    df.to_csv(f"datasets/{symbol}_daily_data.csv", index=False)

    close_prices = df[['Close']].values
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(close_prices)

    split_index = int(len(scaled_data) * 0.8)
    train_data = scaled_data[:split_index]
    test_data = scaled_data[split_index - time_steps:]

    X_train, y_train = create_sequences(train_data, time_steps)
    X_test, y_test = create_sequences(test_data, time_steps)

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.1),
        LSTM(32),
        Dropout(0.1),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mean_squared_error')

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    checkpoint_path = f'models/{symbol}_best_model.h5'
    checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, checkpoint],
        verbose=0
    )

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_preds_inv = scaler.inverse_transform(train_preds)
    test_preds_inv = scaler.inverse_transform(test_preds)
    y_train_inv = scaler.inverse_transform(y_train)
    y_test_inv = scaler.inverse_transform(y_test)

    print_metrics_inline("Train", y_train_inv, train_preds_inv)
    print_metrics_inline("Test", y_test_inv, test_preds_inv)

    # === Plot Test Set Predictions ===
    plt.figure(figsize=(12, 6))
    plt.plot(y_test_inv, label='Actual Price')
    plt.plot(test_preds_inv, label='Predicted Price')
    plt.title(f'{symbol} Stock Price Prediction (Test Set)')
    plt.xlabel('Time Steps')
    plt.ylabel('Price (USD)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'test_set_predictions/{symbol}_test_plot.png')
    plt.close()

    # === Plot Training Loss ===
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{symbol} - Training vs Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'training_loss/{symbol}_loss_plot.png')
    plt.close()

    # === Predict Next Day Price ===
    last_60 = scaled_data[-time_steps:]
    last_60 = np.expand_dims(last_60, axis=0)

    next_day_scaled = model.predict(last_60)
    next_day_price = scaler.inverse_transform(next_day_scaled)

    print(f"Predicted next day's closing price for {symbol}: ${next_day_price[0][0]:.2f}")


    # Metrics
    # Inverse transform predictions and true values
    train_preds_inv = scaler.inverse_transform(train_preds)
    test_preds_inv = scaler.inverse_transform(test_preds)
    y_train_inv = scaler.inverse_transform(y_train)
    y_test_inv = scaler.inverse_transform(y_test)

    # Calculate metrics
    mae_train = mean_absolute_error(y_train_inv, train_preds_inv)
    mse_train = mean_squared_error(y_train_inv, train_preds_inv)
    r2_train = r2_score(y_train_inv, train_preds_inv)

    mae_test = mean_absolute_error(y_test_inv, test_preds_inv)
    mse_test = mean_squared_error(y_test_inv, test_preds_inv)
    r2_test = r2_score(y_test_inv, test_preds_inv)

    results = {
    "symbol": symbol,
    "train_mae": mae_train,
    "train_mse": mse_train,
    "train_r2": r2_train,
    "test_mae": mae_test,
    "test_mse": mse_test,
    "test_r2": r2_test,
    "next_day_prediction": next_day_price[0][0]
    }
    df_results = pd.DataFrame([results])
    df_results.to_csv(f"metrics/{symbol}_model_metrics.csv", mode='a', header=not os.path.exists("model_metrics.csv"), index=False)

    


/tmp/ipykernel_14470/1508690791.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date, interval='1d').reset_index()
[*********************100%***********************]  1 of 1 completed


=== Processing UNH ===



/workspaces/Stock-Market-Prediction-AI-Model/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Train MAE: 3.1849, MSE: 19.2242, R²: 0.9961
Test MAE: 9.8497, MSE: 216.6028, R²: 0.9703
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted next day's closing price for UNH: $312.40
